# Bank Customer Churn Prediction using Neural Network

## Objective
Build a neural-network based binary classifier to predict whether a bank customer will leave the bank (`Exited = 1`) or stay (`Exited = 0`) in the next 6 months.

## Dataset
The Kaggle Bank Customer Churn Modeling dataset contains 10,000 customer records. The target column is `Exited`. Identifier columns such as `RowNumber`, `CustomerId`, and `Surname` are removed because they do not provide useful predictive information.

## Tasks
1. Read the dataset.
2. Separate features and target and split into training and testing sets.
3. Normalize the train and test data.
4. Build a neural network.
5. Improve the model using Dropout and Early Stopping.
6. Print accuracy score and confusion matrix.


### Theory
We import the libraries needed to read data, build a neural network, and measure its performance.

In [1]:
# Step 1: Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'tensorflow'

In [2]:
%pip install tensorflow


[notice] A new release of pip is available: 26.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: Exception:
Traceback (most recent call last):
  File "C:\Users\HARSH\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 438, in _error_catcher
    yield
  File "C:\Users\HARSH\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 561, in read
    data = self._fp_read(amt) if not fp_closed else b""
  File "C:\Users\HARSH\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\urllib3\response.py", line 527, in _fp_read
    return self._fp.read(amt) if amt is not None else self._fp.read()
  File "C:\Users\HARSH\AppData\Local\Programs\Python\Python310\lib\site-packages\pip\_vendor\cachecontrol\filewrapper.py", line 104, in read
    self.__buf.write(data)
  File "C:\Users\HARSH\AppData\Local\Programs\Python\Python310\lib\tempfile.py", line 479, in func_wrap


   ---------------------------------------- 0.0/350.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/350.6 MB ? eta -:--:--
   ---------------------------------------- 0.8/350.6 MB 2.8 MB/s eta 0:02:06
   ---------------------------------------- 1.0/350.6 MB 1.9 MB/s eta 0:03:01
   ---------------------------------------- 1.3/350.6 MB 2.0 MB/s eta 0:02:58
   ---------------------------------------- 2.1/350.6 MB 2.3 MB/s eta 0:02:32
   ---------------------------------------- 2.9/350.6 MB 2.6 MB/s eta 0:02:13
   ---------------------------------------- 3.9/350.6 MB 3.0 MB/s eta 0:01:56
    --------------------------------------- 5.0/350.6 MB 3.4 MB/s eta 0:01:42
    --------------------------------------- 6.3/350.6 MB 3.7 MB/s eta 0:01:33
    --------------------------------------- 7.3/350.6 MB 3.8 MB/s eta 0:01:30
    --------------------------------------- 8.7/350.6 MB 4.1 MB/s eta 0:01:24
   - -------------------------------------- 9.7/350.6 MB 4.2 MB/s eta 0:01:22

### Theory
The CSV file contains bank customer details. We load it into a DataFrame so we can inspect and process it.

In [ ]:
# Step 2: Read Dataset
# Keep Churn_Modelling.csv in the same folder as this notebook.
df = pd.read_csv("Churn_Modelling.csv")
print(df.shape)
df.head()

### Theory
We check the dataset structure, missing values, and the number of customers who stayed or left.

In [ ]:
# Basic Dataset Information
print(df.info())
print("\nMissing values:")
print(df.isnull().sum())
print("\nChurn count:")
print(df["Exited"].value_counts())

### Theory
We remove identifier columns and convert text categories into numeric values because a neural network works with numbers.

In [ ]:
# Step 3: Preprocessing
# Remove columns that do not help prediction.
df = df.drop(["RowNumber", "CustomerId", "Surname"], axis=1)

# Convert text values into numbers.
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=True)

df.head()

### Theory
Features (X) are the customer details used for prediction. Target (y) is the Exited column we want to predict.

In [ ]:
# Step 4: Separate Features and Target
X = df.drop("Exited", axis=1)  # Input data
y = df["Exited"]               # Output to predict

print(X.shape, y.shape)

### Theory
We split the data into training and testing parts. The model learns from training data and is checked on testing data.

In [ ]:
# Step 5: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

### Theory
Scaling brings numeric features to a similar range, helping the neural network train more effectively.

In [ ]:
# Step 6: Normalize Data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Data normalized!")

## Step 7: Build the Initial Neural Network

The initial model uses:
- Input layer
- Two hidden layers with ReLU activation
- Output layer with Sigmoid activation
- Adam optimizer
- Binary cross-entropy loss

Since this is binary classification, sigmoid is used in the output layer.


### Theory
A neural network has input, hidden, and output layers. ReLU is used in hidden layers and sigmoid gives the churn probability.

In [ ]:
# Initial Neural Network
model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

### Theory
The model learns patterns from training data over several epochs. We then check its accuracy on test data.

In [ ]:
# Train Initial Model
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_split=0.2)

loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test Accuracy:", round(accuracy * 100, 2), "%")

## Step 8: Points of Improvement

The basic neural network can overfit or continue training after validation performance stops improving.

### Improvements implemented
1. Increase hidden-layer capacity.
2. Add Dropout to reduce overfitting.
3. Use Early Stopping to stop training when validation loss stops improving.
4. Restore the best model weights.


### Theory
Dropout randomly disables some neurons during training. This helps reduce overfitting.

In [ ]:
# Improved Neural Network
improved_model = Sequential([
    Dense(16, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(8, activation="relu"),
    Dense(1, activation="sigmoid")
])

improved_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
improved_model.summary()

### Theory
Early Stopping stops training when validation loss stops improving and restores the best model weights.

In [ ]:
# Early Stopping and Training
early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

history = improved_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop]
)

### Theory
The model predicts a probability for each customer. Values 0.5 or higher are classified as churn (1); otherwise, stay (0).

In [ ]:
# Step 9: Predict Test Data
y_prob = improved_model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int).ravel()
print("Prediction completed!")

### Theory
Accuracy is the percentage of test predictions that are correct.

In [ ]:
# Step 10: Accuracy Score
print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")

### Theory
The confusion matrix compares actual and predicted classes, showing correct and incorrect predictions.

In [ ]:
# Step 11: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

sns.heatmap(cm, annot=True, fmt="d")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Theory
The classification report displays precision, recall, and F1-score for each class.

In [ ]:
# Classification Report
print(classification_report(y_test, y_pred))

### Theory
This graph compares training accuracy with validation accuracy across epochs.

In [ ]:
# Training and Validation Accuracy
plt.plot(history.history["accuracy"], label="Training")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

### Theory
This graph compares training loss with validation loss across epochs.

In [ ]:
# Training and Validation Loss
plt.plot(history.history["loss"], label="Training")
plt.plot(history.history["val_loss"], label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

## Conclusion

The bank customer churn dataset was preprocessed by removing irrelevant identifier columns and encoding categorical features. The data was split into training and testing sets and normalized using StandardScaler.

A neural network classifier was developed using ReLU hidden layers and a sigmoid output layer. The model was improved using Dropout and Early Stopping to reduce overfitting.

The final model was evaluated using accuracy and a confusion matrix.
